In [1]:
import os
os.environ["GEMINI_API_KEY"]= "<API KEY>"

In [2]:
from PIL import Image
import os

folder = "artemis_flyby_images"
if not os.path.exists(folder):
    os.makedirs(folder)

files = [f for f in os.listdir(folder) if f.endswith(('.jpg', '.jpeg', '.png'))]

print(f"Found {len(files)} image files:\n")
for f in files[:8]:   # show first 8
    try:
        path = os.path.join(folder, f)
        with Image.open(path) as img:
            print(f"✅ {f}  →  {img.format} | {img.size} | {img.mode}")
    except Exception as e:
        print(f"❌ {f}  →  ERROR: {e}")

Found 13 image files:

✅ art002e009006.jpg  →  JPEG | (1920, 1280) | RGB
✅ art002e009562.jpg  →  JPEG | (1920, 1440) | RGB
✅ art002e009206.jpg  →  JPEG | (1920, 1280) | RGB
✅ art002e009289.jpg  →  JPEG | (1920, 1280) | RGB
✅ art002e009287.jpg  →  JPEG | (1920, 1280) | RGB
✅ art002e009298.jpg  →  JPEG | (1920, 1280) | RGB
✅ art002e004438.jpg  →  JPEG | (1920, 1280) | RGB
✅ art002e009276.jpg  →  JPEG | (1920, 1280) | RGB


In [3]:
!pip install google-genai

#### =============With Gemini Flash model==============

In [4]:
import google.generativeai as genai
from PIL import Image

genai.configure(api_key=os.environ["GEMINI_API_KEY"])
model = genai.GenerativeModel("gemini-3-flash-preview")

def analyze_image(image_path, custom_prompt=""):
    img = Image.open(image_path)

    prompt = custom_prompt or """Analyze this Artemis II lunar flyby image from April 6, 2026.
    1. Describe the visible geological features (craters, basins like Orientale, maria, etc.)
    2. Explain their scientific significance
    3. Note anything unique (far-side view, Earthrise, first-time human observation)"""

    response = model.generate_content([prompt, img])
    print("🔍 ANALYSIS:\n")
    print(response.text)

# Test on the first image
image_files = [f for f in os.listdir("artemis_flyby_images") if f.endswith(".jpg")]
test_image = f"artemis_flyby_images/{image_files[0]}"
print(f"Analyzing: {image_files[0]}")
analyze_image(test_image)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Analyzing: art002e009006.jpg
🔍 ANALYSIS:

Based on a visual analysis of the image provided, there is a significant discrepancy between your description and the actual content of the photo.

First, a correction: **This is a standard image of the Moon's near side (the side facing Earth) at nearly full phase.** It is not a high-resolution, close-up flyby image from the future Artemis II mission (scheduled for late 2024 or 2025). It appears to be a photograph taken from Earth with a telephoto lens.

Here is the analysis based on the features actually visible in the image you provided:

### 1. Visible Geological Features

The image shows the familiar "Man in the Moon" face that we see from Earth.

*   **Maria (Lunar "Seas"):** These are the large, dark, relatively flat plains. They are not water, but ancient beds of solidified basaltic lava.
    *   **Oceanus Procellarum (Ocean of Storms):** The massive dark area covering most of the left (western) side.
    *   **Mare Imbrium (Sea of Rains

In [11]:
def analyze_lunar_image(image, custom_prompt=""):
    if image is None:
        return "⚠️ Please select an image or upload one."

    # IMPORTANT FIX: Convert image to RGB mode (removes alpha channel)
    try:
        if image.mode in ("RGBA", "P", "LA"):
            image = image.convert("RGB")
        elif image.mode != "RGB":
            image = image.convert("RGB")
    except:
        image = image.convert("RGB")  # fallback

    default_prompt = """You are a passionate lunar geologist and space enthusiast.
Analyze this image from NASA's Artemis II lunar flyby on April 6, 2026.

Provide a clear, detailed, and exciting analysis with these points:
1. Main geological features visible (craters, basins such as Orientale, maria, highlands, etc.)
2. Scientific significance and why this view is important
3. Unique aspects from this mission (far-side view, first human observation from Orion, Earthrise, etc.)
4. How this data helps prepare for future Artemis landings and eventual Mars missions

Use an enthusiastic but professional tone. Keep it concise yet informative (around 250-350 words)."""

    prompt = custom_prompt.strip() if custom_prompt.strip() else default_prompt

    try:
        response = gemini_model.generate_content([prompt, image])
        return response.text
    except Exception as e:
        return f"❌ Analysis error: {str(e)}"

In [ ]:
folder = "artemis_flyby_images"
image_files = []
if os.path.exists(folder):
    image_files = sorted([f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

with gr.Blocks(title="Artemis II Lunar Flyby Analyzer", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🌕 Artemis II Lunar Flyby Image Analyzer")
    gr.Markdown("**Real images from April 6, 2026 lunar flyby** • Powered by Gemini 3 Flash Preview")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Choose an Image")
            image_dropdown = gr.Dropdown(
                choices=image_files,
                label="Select from your uploaded folder",
                value=image_files[0] if image_files else None,
                interactive=True
            )
            upload_image = gr.Image(type="pil", label="Or upload a new image")

            custom_prompt = gr.Textbox(
                label="Custom Prompt (optional)",
                placeholder="Example: Focus especially on the Orientale Basin or Earth views",
                lines=2
            )
            analyze_btn = gr.Button("🚀 Analyze Image", variant="primary", size="large")

        with gr.Column(scale=2):
            output_box = gr.Markdown(label="🔬 Analysis Result")

    # Load image when dropdown is changed
    def load_image_from_dropdown(filename):
        if filename and os.path.exists(os.path.join(folder, filename)):
            return Image.open(os.path.join(folder, filename))
        return None

    image_dropdown.change(
        fn=load_image_from_dropdown,
        inputs=image_dropdown,
        outputs=upload_image
    )

    # Analyze button action
    analyze_btn.click(
        fn=analyze_lunar_image,
        inputs=[upload_image, custom_prompt],
        outputs=output_box
    )

    gr.Markdown("""
    **Tips for best results:**
    - Try images of the fully illuminated Moon, Orientale Basin, or crew window views from today's flyby
    - You can add your own custom prompt for specific focus areas
    """)

demo.launch(share=True, debug=True)

/tmp/ipykernel_5830/997209384.py:7: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Artemis II Lunar Flyby Analyzer", theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://926f8b05632261a2b4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Original image mode: RGB, size: (1920, 1280)


ERROR:tornado.access:503 POST /v1beta/models/gemini-3-flash-preview:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 7134.17ms


✅ Gemini API call successful
Original image mode: RGB, size: (1920, 1280)
✅ Gemini API call successful
Original image mode: RGB, size: (1920, 1280)
✅ Gemini API call successful
Original image mode: RGB, size: (1920, 1280)
✅ Gemini API call successful


#### ===== With Gemini Flash & Florence-2 models ====

In [13]:
!pip install -U gradio pillow transformers accelerate --quiet

In [15]:
!pip uninstall -y Pillow pillow
!pip install Pillow==11.1.0 --quiet
!pip install transformers==4.45.2 --quiet
!pip install accelerate --quiet
print("✅ Pillow downgraded and transformers reinstalled. Restart runtime now!")

Found existing installation: pillow 11.1.0
Uninstalling pillow-11.1.0:
  Successfully uninstalled pillow-11.1.0
✅ Pillow downgraded and transformers reinstalled. Restart runtime now!


In [6]:
import google.generativeai as genai
from PIL import Image
import gradio as gr
import os
import torch
from transformers import AutoProcessor, AutoModelForCausalLM

print("Pillow Version:", Image.__version__)

# ====================== GEMINI SETUP ======================
GEMINI_API_KEY = "<API KEY>"
genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel("gemini-3-flash-preview")

# ====================== FLORENCE-2 SETUP ======================
print("\nLoading Florence-2 (Hugging Face)... This may take 30-60 seconds...")
florence_available = False

try:
    florence_processor = AutoProcessor.from_pretrained("microsoft/Florence-2-large", trust_remote_code=True)
    florence_model = AutoModelForCausalLM.from_pretrained("microsoft/Florence-2-large", trust_remote_code=True)
    florence_device = "cuda" if torch.cuda.is_available() else "cpu"
    florence_model = florence_model.to(florence_device)
    florence_available = True
    print("✅ Florence-2 loaded successfully!")
except Exception as e:
    print(f"❌ Florence-2 failed to load: {e}")

Pillow Version: 11.3.0

Loading Florence-2 (Hugging Face)... This may take 30-60 seconds...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

processing_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-large:
- processing_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

modeling_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-large:
- modeling_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

✅ Florence-2 loaded successfully!


In [9]:
def analyze_with_gemini(image):
    if image is None:
        return "No image provided."
    if image.mode != "RGB":
        image = image.convert("RGB")

    prompt = """Analyze this Artemis II lunar flyby image taken on April 6, 2026 from the Orion spacecraft.
    Provide a clear, structured, and enthusiastic scientific description of the geological features."""

    try:
        response = gemini_model.generate_content([prompt, image])
        return response.text
    except Exception as e:
        return f"Gemini Error: {str(e)}"


def analyze_with_florence(image):
    if not florence_available or image is None:
        return "Florence-2 is not available."
    if image.mode != "RGB":
        image = image.convert("RGB")

    try:
        prompt = "Act as a lunar geologist, describe this image in detail, especially the moon's surface features, craters, and terrain."
        inputs = florence_processor(text=prompt, images=image, return_tensors="pt").to(florence_device)
        generated_ids = florence_model.generate(**inputs, max_new_tokens=512)
        result = florence_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        return result
    except Exception as e:
        return f"Florence-2 Error: {str(e)}"

In [12]:
folder = "artemis_flyby_images"
image_files = sorted([f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]) if os.path.exists(folder) else []

with gr.Blocks(title="Artemis II Image Analyzer", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🌕 Artemis II Lunar Flyby Image Analyzer")
    gr.Markdown("**Gemini 3 Flash Preview vs Florence-2 (Hugging Face)** — April 6, 2026 Mission Images")

    with gr.Row():
        with gr.Column(scale=1):
            image_dropdown = gr.Dropdown(
                choices=image_files,
                label="Select Image from Folder",
                value=image_files[0] if image_files else None
            )
            upload_image = gr.Image(type="pil", label="Or Upload New Image")
            analyze_btn = gr.Button("🚀 Analyze with Both Models", variant="primary", size="large")

        with gr.Column(scale=2):
            gr.Markdown("### 🔬 <span style='color:blue'>Gemini 3 Flash Preview</span>")
            gemini_output = gr.Markdown()
            gr.Markdown("### 🌟 <span style='color:red'>Florence-2 (Hugging Face)</span>")
            florence_output = gr.Markdown()

    # Load image from dropdown
    def load_image(filename):
        if filename:
            return Image.open(os.path.join(folder, filename))
        return None

    image_dropdown.change(fn=load_image, inputs=image_dropdown, outputs=upload_image)

    # Analyze button
    def analyze_both(image):
        gemini_result = analyze_with_gemini(image)
        florence_result = analyze_with_florence(image)
        return gemini_result, florence_result

    analyze_btn.click(
        fn=analyze_both,
        inputs=upload_image,
        outputs=[gemini_output, florence_output]
    )

    gr.Markdown("**Note**: Gemini usually gives better scientific reasoning. Florence-2 is good at dense visual description.")

demo.launch(share=True, debug=True)

/tmp/ipykernel_4512/3717033435.py:5: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Artemis II Image Analyzer", theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://bdd7125c69ebd4b801.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://bdd7125c69ebd4b801.gradio.live


##### ======== With Qwen2.5-VL-7B model=======

In [3]:
!pip install git+https://github.com/huggingface/transformers.git --quiet
!pip install qwen-vl-utils --quiet

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 20.3 MB/s eta 0:00:00


In [1]:
import google.generativeai as genai
from PIL import Image
import gradio as gr
import os
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

print("✅ Imports successful!")

# Gemini Setup
GEMINI_API_KEY = "<API KEY>"
genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel("gemini-3-flash-preview")

# Qwen2.5-VL Setup
print("Loading Qwen2.5-VL-7B-Instruct...")
qwen_available = False

try:
    qwen_processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct", trust_remote_code=True)
    qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        "Qwen/Qwen2.5-VL-7B-Instruct",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
        trust_remote_code=True
    )
    qwen_available = True
    print("✅ Qwen2.5-VL-7B loaded successfully!")
except Exception as e:
    print(f"❌ Qwen loading error: {e}")
    qwen_available = False



/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ Imports successful!
Loading Qwen2.5-VL-7B-Instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

✅ Qwen2.5-VL-7B loaded successfully!


In [8]:
def analyze_with_gemini(image):
    if image is None:
        return "No image provided."
    if image.mode != "RGB":
        image = image.convert("RGB")

    prompt = """You are a lunar geologist. Analyze this Artemis II lunar flyby image from April 6, 2026.
Describe the main geological features (craters, basins like Orientale, maria, highlands), lighting, and scientific significance."""

    try:
        response = gemini_model.generate_content([prompt, image])
        return response.text
    except Exception as e:
        return f"Gemini Error: {str(e)}"


def analyze_with_qwen(image):
  # ====================== Qwen2.5-VL-3B (Light Version ) ======================
  print("Loading Qwen2.5-VL-3B-Instruct (lighter version to avoid OOM)...")

  qwen_available = False

  try:
      qwen_processor = AutoProcessor.from_pretrained(
          "Qwen/Qwen2.5-VL-3B-Instruct",
          trust_remote_code=True
      )

      qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
          "Qwen/Qwen2.5-VL-3B-Instruct",
          torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
          device_map="auto",
          trust_remote_code=True
      )

      qwen_available = True
      print("✅ Qwen2.5-VL-3B loaded successfully! (Much lighter version)")

  except Exception as e:
      print(f"❌ Qwen loading failed: {e}")
      qwen_available = False


  '''   if not qwen_available or image is None:
        return "Qwen2.5-VL is not available."
    if image.mode != "RGB":
        image = image.convert("RGB")

    try:
        messages = [
            {"role": "user", "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": "You are a lunar geologist. Provide a detailed analysis of this Artemis II lunar flyby image. Focus on geological features, craters, basins, lighting conditions, and scientific importance."}
            ]}
        ]

        text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = qwen_processor(text=[text], images=[image], padding=True, return_tensors="pt").to(qwen_model.device)

        generated_ids = qwen_model.generate(**inputs, max_new_tokens=600)
        generated_text = qwen_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        return generated_text
    except Exception as e:
        return f"Qwen2.5-VL Error: {str(e)}" '''


print("✅ Analysis functions defined successfully!")

✅ Analysis functions defined successfully!


In [ ]:
with gr.Blocks(title="Artemis II Lunar Flyby Analyzer", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🌕 Artemis II Lunar Flyby Image Analyzer")
    gr.Markdown("**Gemini 3 Flash Preview vs Qwen2.5-VL-7B** — Real images from April 6, 2026 lunar flyby")

    with gr.Row():
        with gr.Column(scale=1):
            # Image selection
            folder = "artemis_flyby_images"
            image_files = sorted([f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]) if os.path.exists(folder) else []

            image_dropdown = gr.Dropdown(
                choices=image_files,
                label="Select Image from Folder",
                value=image_files[0] if image_files else None
            )
            upload_image = gr.Image(type="pil", label="Or Upload New Image")
            analyze_btn = gr.Button("🚀 Analyze with Both Models", variant="primary", size="large")

        with gr.Column(scale=2):
            gr.Markdown("### 🔬<span style='color:blue'>Gemini 3 Flash Preview</span>")
            gemini_output = gr.Markdown()

            gr.Markdown("### 🌟<span style='color:red'>Qwen2.5-VL-7B (Open Source)</span>")
            qwen_output = gr.Markdown()


    def load_selected_image(filename):
        if filename and os.path.exists(os.path.join(folder, filename)):
            return Image.open(os.path.join(folder, filename))
        return None

    image_dropdown.change(fn=load_selected_image, inputs=image_dropdown, outputs=upload_image)

    # Analysis function
    def analyze_both(image):
        if image is None:
            return "Please select or upload an image.", "Please select or upload an image."

        gemini_result = analyze_with_gemini(image)
        qwen_result = analyze_with_qwen(image)

        return gemini_result, qwen_result

    analyze_btn.click(
        fn=analyze_both,
        inputs=upload_image,
        outputs=[gemini_output, qwen_output]
    )

    gr.Markdown("""
    **Comparison Note**:
    - Gemini is usually better at scientific reasoning and mission context.
    - Qwen2.5-VL is fully open-source and often gives very detailed visual descriptions.
    """)

demo.launch(share=True, debug=True)

/tmp/ipykernel_28478/1391992376.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Artemis II Lunar Flyby Analyzer", theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://155caefb8fd44e5a50.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:tornado.access:503 POST /v1beta/models/gemini-3-flash-preview:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 42502.03ms


Loading Qwen2.5-VL-3B-Instruct (lighter version to avoid OOM)...


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]